크롤링시 소요되는 시간이 아깝다.. 데이터 노이즈도 섞여있고..
크롤링은 충분히 연습했다고 생각하고, 캐글 오픈 데이터셋을 사용하기로 결정.

In [1]:
!pwd

/Users/jeongjaehun/Crawling/02_crawling_data


In [19]:
import os
from sklearn.model_selection import train_test_split
import pandas as pd

# apple: 6668(2819 + 3849)
# banana: 6311(2849 + 3462)
# orange: 3852(1854 + 1998)
# total: 16831

# path = "images/kaggle_data/dataset/apple"
# categories = ["fresh_apple", "rotten_apple"]

data = []
fruits = ["apple", "banana", "orange"]
for fruit in fruits:
    path = f"images/kaggle_data/dataset/{fruit}"
    categories = [f"fresh_{fruit}", f"rotten_{fruit}"]

    for category in categories:
        folder_path = os.path.join(path, category)
        for image in os.listdir(folder_path):
            if image.endswith(("jpg", "jpeg", "png")):
                data.append({
                    'image_path': os.path.join(folder_path, image), 
                    'label_name': category
                })

df = pd.DataFrame(data)
# print(len(df)) # 16831

categories = []
for fruit in fruits:
    categories.append(f"fresh_{fruit}")
    categories.append(f"rotten_{fruit}")

# print(categories)
label_map = {name: i for i, name in enumerate(categories)}

df['label'] = df['label_name'].map(label_map)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
train_df.reset_index(drop=True)
test_df.reset_index(drop=True)

print(f"Train Count: {len(train_df)}, Test Count: {len(test_df)}")

Train Count: 13464, Test Count: 3367


In [23]:
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from torch.utils.data import Dataset, DataLoader

train_transform = A.Compose([
    A.Resize(128, 128),
    A.HorizontalFlip(p=0.5), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])

test_transform = A.Compose([
    A.Resize(128, 128),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])

class FruitDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = row['image_path']
        image = cv2.imread(image)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            aug = self.transform(image=image)
            image = aug['image']
            
        label = int(row['label'])
        return image, torch.tensor(label, dtype=torch.long)

train_dataset = FruitDataset(df=train_df, transform=train_transform)
test_dataset = FruitDataset(df=test_df, transform=test_transform)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [26]:
images, labels = next(iter(train_dataloader))
print(images.shape)
print(labels.shape)

torch.Size([32, 3, 128, 128])
torch.Size([32])


In [29]:
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # (32, 3, 128, 128) -> (32, 8, 64, 64) -> (32, 16, 32, 32) -> (32, 32, 16, 16)
        # -> (32, 64, 8, 8)
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(8)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(16)
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 6)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)

In [30]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [31]:
epochs = 5
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for images, labels in train_dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_dataloader.dataset):.4f}")

model.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()

print(f"Test Result | Accuracy: {correct/len(test_dataloader.dataset) * 100:.2f}%")

Epoch 1/5 | Train Loss: 0.3955
Epoch 2/5 | Train Loss: 0.1853
Epoch 3/5 | Train Loss: 0.1443
Epoch 4/5 | Train Loss: 0.1048
Epoch 5/5 | Train Loss: 0.0975
Test Result | Accuracy: 97.45%


In [32]:
print(model)

SimpleCNN(
  (conv1): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (fc1): Linear(in_features=4096, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=6, bias=True)
)


In [34]:
torch.save(model.state_dict(), 'my_custom_cnn_fruits.pth')